# SPOD to Mapping Excel
- Prerequisites: 
  - Anaconda packages: `xlsxwriter` pandas, openpyxl, seaborn`


## Result

Excel sheet containing:

Sheet with all Datapoints (Databases, Tables and Columns) mapped against the Information Model (Entity, Attribute)
Table covering the overview sheet

Analog dev_x_mapping

Optional:
Sheet per System - IM containing sample data

## Structure

1. Define mapping between SPOD (json) and columns in the resulting Excel sheet
1. Use row emitter to iterate the whole sheet
    1. Fill cells
    1. Style cells
    1. Protect cells
1. Style whole sheets. Group or hide columns.

## Configuration
The following parameters need to be definded when running as regular python script

In [ ]:
LIBRARY = '../../pythonWork/pythonSource'

DESTINATION = 'SIKA_mapping_2022-07-06.xlsx'

MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.json'
#MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.import.regen.json'

## Check prerequisites

In [ ]:
import sys
import logging
import os
import json

In [ ]:
import re
import collections
from pathlib import Path

In [ ]:
# openpyxl
from openpyxl import Workbook
from openpyxl.worksheet.table import Table
from openpyxl.utils.cell import get_column_letter
from openpyxl.styles import PatternFill

In [ ]:
from tqdm.autonotebook import tqdm

In [ ]:
spod_file = Path(MODEL_SOURCE)
assert spod_file.is_file(), f"Cannot find SPOD file '{spod_file.resolve()}'"

with open(spod_file, 'r') as src:
    spod = json.load(src)
assert spod['model'] is not None
print(f"Loaded SPOD containing {spod['model']} from '{spod_file.resolve()}'")

In [ ]:
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
print(f"Languages: {list(spod['languages'].keys())}")
mapdict = {}
for entry in ['entities', 'attributes', 'databases', 'columns']:
    appendix = ''
    if 'columns' == entry:
        mapped_columns_count = len(list(filter(lambda c: len(c['attributesmapped']) > 0, spod[entry].values())))
        appendix = f" (mapped {mapped_columns_count})"
    print(f"- {entry}: {len(spod[entry])}{appendix}")

## Initialize logging

In [ ]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/sharepoint-list-sync-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

urlliblogger = logging.getLogger('urllib3.connectionpool')
urlliblogger.setLevel(logging.DEBUG)

## Use the fyayc SPOD library

In [ ]:
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

from PUBLISH_MODEL.excel.mapping_publisher import generate
from SSOT_infra.translator import Translator

## Translation shortcut tr

In [ ]:
translator = Translator('de')

## List structure definition

In [ ]:
system_index = {}

In [ ]:
headings_im = [
    "FQN", "EID", "AID", 
    "# Sources", "# Sinks", "Mandatory/Optional", 
    "Level",  # Sika specific
]

translations_start = len(headings_im)

for lang in spod['languages'].keys():
    headings_im.append(f"Entity ({lang})")
    headings_im.append(f"Attribute ({lang})")
    headings_im.append(f"Description ({lang})")
    headings_im.append(f"Examples ({lang})")
    headings_im.append(f"{lang}")
    
headings_im.append('|')
headings_im.append('Source')

In [ ]:
translations_start, len(headings_im)

In [ ]:
headings_im

In [ ]:
def prefix_sources(system: tuple) -> str:
    """ Return system name with a A{nr}-src: prefix"""
    sources = ['SAP-ERP', 'SAP.+', 'Hybris.*', 'Stibo.*', 'PIM.*', 'CXM.+' ]
    name = system[1]['name']
    index = 0
    for entry in sources:
        index += 1
        if re.match(entry, name):
            return chr(ord('A') + index) + '-src: ' + name, True
    return chr(ord('Z') + index) + name, False

In [ ]:
databases = collections.OrderedDict(sorted(filter(lambda t: not t[1]['name'].startswith('reserve'), spod['databases'].items()), key=prefix_sources))
skeys = databases.keys()
list(map(lambda t: (t[0], t[1]['name'], len(list(filter(lambda c: c['database-id+'] == t[0], spod['columns'].values())))), databases.items()))

In [ ]:
def emit_system_columns(databases: iter) -> [str]:
    result = []
    for key, system in databases.items():
        system_index[key] = len(headings_im) + len(result)
        result.append(system['name'])
    return result

In [ ]:
databases_headings = emit_system_columns(databases)

In [ ]:
logging.info(f"Mapping {len(systems_headings)} systems")

In [ ]:
systems_headings

In [ ]:
def export_column(key: str, column: dict) -> []:
    result = [
            column['name'],
    ]
    return result

In [ ]:
headings = headings_im + systems_headings
f"Columns ({len(headings)}): {', '.join(headings)}"

## Append unmapped columns to the bottom

# Prepare output formatting

In [ ]:
import xlsxwriter

In [ ]:
xlsx_destination = Path(DESTINATION)
workbook = xlsxwriter.Workbook(xlsx_destination)

### Cell formats

In [ ]:
title_format = workbook.add_format({'bold': True, 'font_color': 'black', 'font_size': 20})
column_head_format = workbook.add_format({'bold': True, 'bg_color': '#A0A0A0'})

In [ ]:
# light red background, red font as with Excel 'bad'
cell_bad = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'})
# green background, dark green font as with Excel 'good'
good_style = {'bg_color': '#006100', 'font_color': '#C6EFCE'}
cell_good = workbook.add_format(good_style)

cell_yellow_background = workbook.add_format({'bg_color': '#F7F306', 'font_color': '#000000'})
cell_green_background = workbook.add_format({'bg_color': '#BAEEBA', 'font_color': '#FFFFFF'})

# Gray background for non-mandatory cells
cell_grey_background = workbook.add_format({'bg_color': '#E0E0E0', 'font_color': '#808080' })

In [ ]:
title_format = workbook.add_format({'bold': True, 'font_color': 'black', 'font_size': 20})

column_head_format = workbook.add_format({'bold': True, 'bg_color': '#A0A0A0'})

cell_default_format = workbook.add_format({})

level_colors = [
    workbook.add_format({'bg_color': '#FAEC2D'}),
    workbook.add_format({'bg_color': '#FAAC0D'}),
    workbook.add_format({'bg_color': '#9A9A9A'}),
    workbook.add_format({'bg_color': '#AB8D6D'}),
    workbook.add_format({'bg_color': '#008D00'}),
]

unleveled = len(level_colors) + 1
unleveled

In [ ]:
## Summary is first sheet, but will be filled last
summary = workbook.add_worksheet('Summary')

## Create 'Mapping' sheet

In [ ]:
worksheet = workbook.add_worksheet('Mapping')

### Sort by Attribute FQN
#data_table.sort(key=lambda r: r[0] if r[0] is not None else '\uFFFF')


### Headers

In [ ]:
col = 0
for header in headings:
    worksheet.write(0, col, header, title_format)
    col += 1

### Mapped columns

In [ ]:
columns_mapped = {}

In [ ]:
def get_sika_level(column: dict) -> int:
    mappings = column['userdefprops'].get('column-mapping', {}).get('EXTERNAL', {})
    for mapping in mappings.values():
        value = mapping.get('value')
        #print(f"{mapping} = {value}")
        if mapping.get('name') == 'SIKA-PACKAGING-LEVEL' and value is not None:
            try:
                return int(value)
            except ValueError:
                pass
    return unleveled

In [ ]:
get_sika_level(next(iter(map(lambda t: t[1], filter(lambda t: 'TYPE_SURCOND' in t[1]['name'], spod['columns'].items())))))

In [ ]:
leveled = list(filter(lambda c: get_sika_level(c) != unleveled, spod['columns'].values()))
len(leveled)

In [ ]:
leveled_without_mapping = list(filter(lambda c: len(c['attributesmapped']) < 1, leveled))
len(leveled_without_mapping)

In [ ]:
logging.info(f"Sika annotated columns: {len(leveled_without_mapping)}/{len(leveled)}")

In [ ]:
def is_mapped_on_level(spod: dict, column_key: str, attribute_key: str, level: int) -> bool:
    column = spod['columns'][column_key]
    if attribute_key in column['attributesmapped']:
        if level is not None:
            return level == get_sika_level(column)
        else:
            return get_sika_level(column) == unleveled
    return False

In [ ]:
def full_qualified_column_name(column: dict) -> str:
    return f"{column['database-name+']}.{column['table-name+']}.{column['name']}"

In [ ]:
def all_columns_non_mandatory(spod: dict, column_keys: set) -> bool:
    for ckey in column_keys:
        column = spod['columns'][ckey]
        if column['mandatory']:
            # at least one column is mandatory
            return False
    return True

In [ ]:
def is_source_column(spod: dict, column_key: str) -> bool:
    column = spod['columns'][column_key]
    src = "W" in column['R/W'].upper()
    return src

def src_col_from_attributes(spod: dict, attribute_keys: set) -> dict:
    source_columns = dict(filter(lambda t: is_source_column(spod, t[0]) and len(attribute_keys.intersection(t[1]['attributesmapped'])) > 0, spod['columns'].items()))
    return source_columns


def write_sources(spod: dict, worksheet, row: int, col: int, attribute_keys: set) -> set:
    """Collect source systems for the designated attribute and write a list into the cell"""
    if attribute_keys is None:
        return 0

    source_columns = src_col_from_attributes(spod, attribute_keys)
    lineage = ','.join(map(full_qualified_column_name, source_columns.values()))
    if len(lineage) > 0:
        assert isinstance(lineage, str)
        worksheet.write(row, col, lineage, cell_good)

    return source_columns

In [ ]:
def columns_for_attribute_on_level(spod: dict, attribute_key: str, level: int) -> set:
    """Provide a set of column keys for this level"""
    return set(filter(lambda ckey: is_mapped_on_level(spod, ckey, attribute_key, level), spod['columns'].keys()))

In [ ]:
columns_for_attribute_on_level(spod, 'ATTR8540', None)

In [ ]:
a = spod['attributes']['ATTR8540']
cols = []
for val in a['columnsmapped+'].values():
    cols.extend(val)
cols

In [ ]:
is_mapped_on_level(spod, cols[1], 'ATTR8540', unleveled)

In [ ]:
attribute_key = 'ATTR152'
columns_on_level = columns_for_attribute_on_level(spod, attribute_key, None)
src_cols = src_col_from_attributes(spod, set([attribute_key]))
f"On level {len(columns_on_level)}, out: {len(src_cols)}"

In [ ]:
# Print headings to look up in write_row
index = 0
for title in headings_im:
    print(f"{index:02} {title}")
    index += 1

In [ ]:
def write_system_columns(spod: dict, worksheet, row, col, columns, level) -> [str]:
    """Print the provided columns on the corresponding system's column to this row"""
    
    if level == None or level == unleveled:
        cell_format = None
    else:
        cell_format = level_colors[level]
        
    written = []
    for skey, system in systems.items():
        for ckey in columns:
            if columns_mapped.get(ckey) is None:
                column = spod['columns'][ckey] 
                if column['database-id+'] == skey:
                    
                    style = cell_format
                    
                    if not column['mandatory'] and cell_format is None:
                        style = cell_grey_background
                    columns_mapped[ckey] = column
                    
                    _, is_src_system = prefix_sources( (ckey, system) )
                    if is_src_system:
                        style = cell_yellow_background
                        
                    if 'W' in column['R/W'].upper():
                        style = cell_green_background
                    
                    worksheet.write(row, col, column['name'], style)
                    written.append(ckey)
        col += 1
        
    return written

In [ ]:
systems_start_column = index
logger.info(f"Databases start on column {systems_start_column}")

In [ ]:
non_mandatory_rows = set()

In [ ]:
def write_front(spod: dict, worksheet, row: int, columns: set, level: int, style) -> set:
    col = systems_start_column

    written = write_system_columns(spod, worksheet, row, col, columns, level)
    
    sources = list(filter(lambda c: is_source_column(spod, c), written))
    sinks = list(filter(lambda c: not is_source_column(spod, c), written))
    assert set(sources) | set(sinks) == set(written)
    
    # Numbers of mapped columns
    source_style = cell_green_background if len(written) > 0 else style
    worksheet.write(row, 3, len(sources), source_style)
            
    optional_line = all_columns_non_mandatory(spod, written)
    worksheet.write(row, 4, len(sinks), cell_grey_background if optional_line else style)
    if optional_line:
        non_mandatory_rows.add(row)
        worksheet.write(row, 5, 'optional', cell_grey_background)
    else:
        worksheet.write(row, 5, 'mandatory', style)        
    
    if level is not None and level < len(level_colors):
        l_style = level_colors[level]
    else:
        l_style = None
    worksheet.write(row, 6, None if level is None or level == unleveled else int(level), l_style)

    return written

In [ ]:
def write_row(spod: dict, worksheet, row: int, attribute_key: str, attribute: dict, level: int, translator: Translator) -> []:
    
    if level is not None and level < len(level_colors):
        style = level_colors[level]
    else:
        style = None
            
    enti_key = attribute['entity']
    entity = spod['entities'].get(enti_key)
    assert entity is not None, f"Missing entity {enti_key}"

    col = 0
    worksheet.write(row, col, enti_key + ':' + attribute_key, style)

    enti_cell_style = style
    if entity is None or entity.get('shortname', '') == 'unassigned':
        enti_cell_style = cell_bad
    worksheet.write(row, col + 1, enti_key, enti_cell_style)
    worksheet.write(row, col + 2, attribute_key, enti_cell_style)

    col = translations_start
    for lang in spod['languages'].keys():
        worksheet.write(row, col, translator.tr(entity['name'], lang), enti_cell_style)
        worksheet.write(row, col + 1, translator.tr(attribute['name'], lang), enti_cell_style)
        worksheet.write(row, col + 2, translator.tr(attribute['descr'], lang), style)
        
        exline = ','.join(map(lambda e: translator.tr(e, lang), attribute.get('examples')))
        worksheet.write(row, col + 3, exline, style)
        worksheet.write(row, col + 4, lang, style)
        col += 5

    worksheet.write(row, col, '|')
    
    lineage = write_sources(spod, worksheet, row, col + 1, set( [attribute_key] )) 

    col += 2
    assert col == systems_start_column, f"Column index {col} != {systems_start_column}"

    columns_on_level = columns_for_attribute_on_level(spod, attribute_key, level)
    write_front(spod, worksheet, row, columns_on_level, level, style)
    
    return lineage

In [ ]:
mapped = 0
row = 1

# Iterate levels first
for packaging_level in range(0, len(level_colors)):
    ccount = 0
    for key, attribute in spod['attributes'].items():
        columns_on_level = columns_for_attribute_on_level(spod, key, packaging_level)
        if len(columns_on_level) > 0:
            #logger.info(f"Columns on level {packaging_level} for {key}: {columns_on_level}")
            entry = write_row(spod, worksheet, row, key, attribute, packaging_level, translator)
            mapped += 1 if len(columns_on_level) > 0 else 0
            row += 1        
            ccount += len(columns_on_level)
    logger.info(f"Collected {ccount} for level {packaging_level}")
    
columns_to_go = spod['columns'].keys() - columns_mapped.keys()
logger.info(f"Wrote {row-1} attribute rows. {mapped} are mapped to columns. Unmapped columns remaining: {len(columns_to_go)}. Mapped {len(columns_mapped)}")

In [ ]:
logger.info(f"Non-mandatory rows {non_mandatory_rows}")

### Unmapped attributes

In [ ]:
columns_written = 0

rstart = row
for key, attribute in spod['attributes'].items():
    columns_on_level = columns_for_attribute_on_level(spod, key, unleveled)
    remaining = set(columns_on_level) - set(columns_mapped)
    while len(remaining) > 0:
        #print(f"Adding columns {remaining} for attribute '{key}'")
        columns_on_line = write_row(spod, worksheet, row, key, attribute, unleveled, translator)
        columns_written += len(columns_on_line)
        row += 1
        remaining = set(columns_on_level) - set(columns_mapped)

logger.info(f"Wrote {columns_written} columns on {row - rstart} rows")

### Unmapped columns

In [ ]:
def sort_by_system(ckey: str):
    column = spod['columns'][ckey]
    position = list(systems.keys()).index(column['database-id+'])
    return f"{position:03} {column['name']}"

In [ ]:
def unprocessed():
    return sorted(set(spod['columns'].keys()) - set(columns_mapped.keys()), key=sort_by_system)

In [ ]:
remainder = unprocessed()
logging.info(f"Unprocessed columns remaining {len(remainder)}. Already mapped {len(columns_mapped)}")

In [ ]:
for packaging_level in range(0, len(level_colors)):
    it = iter(unprocessed())
    ckey = next(it, None)
    while ckey is not None:
        column = spod['columns'][ckey]
        clevel = get_sika_level(column)
        if clevel == packaging_level :
            written = write_front(spod, worksheet, row, set([ckey]), packaging_level, None)
            #print(f"Wrote {written} on line {row}")
            row += 1
            it = iter(unprocessed())
        
        ckey = next(it, None)    

In [ ]:
logging.info(f"Processing remainder {len(unprocessed())}")
it = iter(unprocessed())
ckey = next(it, None)
while ckey is not None:
    column = spod['columns'][ckey]
    written = write_front(spod, worksheet, row, set([ckey]), unleveled, None)
    #print(f"Wrote {written} on line {row}")
    row += 1
    it = iter(unprocessed())
    ckey = next(it, None)     

In [ ]:
logger.info(f"Non-mandatory rows {non_mandatory_rows}")

In [ ]:
assert len(columns_mapped) == len(spod['columns']), f"Expecting all columns mapped. Remainder: {columns_mapped}"

### Define Table

In [ ]:
table_column_headers = [ { 'header': name } for name in headings ]

In [ ]:
worksheet.add_table(0, 0, row - 1, len(headings) - 1, { 
    'name': 'mapping',
    'banded_rows': True,
    'columns': table_column_headers,
})

### Styling

In [ ]:
# FQN width
worksheet.set_column(0, 0, 20)

# Hide EID, AID on the left
#worksheet.set_column(1, 3, 10, None, { 'hidden': 1, })

# EID, AID
worksheet.set_column(3, 5, 20)

# Hide attribute name translations (DE, FR)
#worksheet.set_column(5, 8, 40, None, { 'hidden': 1, })

# | Separator
worksheet.set_column(systems_start_column - 2, systems_start_column - 2, 2)

base = systems_start_column
index = 0
for system in skeys:
#    colnr = base + (index * 3)
    colnr = base + index
#    worksheet.set_column(colnr, colnr, None, None, { 'hidden': 1, })
    
    # Column name on system
    worksheet.set_column(colnr, colnr, 30)
    
    # Technical reference
 #   worksheet.set_column(colnr + 2, colnr + 2, None, None, { 'hidden': 1, })        
    index += 1

In [ ]:
# Data bars
worksheet.conditional_format('D2:D' + str(row), {'type': 'data_bar', 'bar_color': good_style['bg_color']})
worksheet.conditional_format('E2:E' + str(row), {'type': 'data_bar'})
# Width
worksheet.set_column('D:E', 8, None)

In [ ]:
# Combine and hide language groups
index = translations_start
for lang in spod['languages'].keys():
    hidden = lang != 'en'
    worksheet.set_column(index, index + 3, 27, None, { 'level': 1, 'hidden': hidden })
    worksheet.set_column(index + 4, index + 4, 1.75, None, { 'collapsed': hidden, })
    index += 5

## Add one sheet per system

In [ ]:
def fill_worksheet(spod: dict, skey: str, system: str, sheet):

    headings = [ 
        'Table Key',
        'Column Key', 
        'Table Name', 
        'Tech-ID', 
        'Name', 
        'Description', 
        'Packaging levevel',
        '|',
        'IM Attributes',
        'Source(s)',
    ]
    
    row = 0
    index = 0
    for heading in headings:
        sheet.write(row, 9, heading, column_head_format)
        index += 1
    
    row += 1
    columns = sorted(list(spod['columns'].items()), key=lambda c: c[1]['table-name+'])
    for ckey, column in columns:
        if column['database-id+'] == skey:
            
            level = get_sika_level(column)
            if level is not None and level < len(level_colors):
                style = level_colors[level]
            else:
                style = None
            
            sheet.write(row, 0, column['table-id'], style)
            sheet.write(row, 1, ckey, style)
            sheet.write(row, 2, column['table-name+'], style)
            sheet.write(row, 3, column['database_col_id'], style)
            sheet.write(row, 4, column['name'], style)
            sheet.write(row, 5, translator.tr(column['descr'], 'de'), style)
            sheet.write(row, 6, level if level < unleveled else None, style)
            sheet.write(row, 7, '|')
            linked_attributes = column['attributesmapped']
            sheet.write(row, 8, ', '.join(linked_attributes))
            lineage = write_sources(spod, sheet, row, 9, set(linked_attributes))
            
            # styling
            sheet.set_column(0, 2, 10, None, {'hidden': True})
            sheet.set_column(1, 9, 45)
            sheet.set_column(5, 5, 70)  # Description
            sheet.set_column(6, 6, 4)  # Packaging level
            sheet.set_column(7, 7, 1.5)  # | Separator
            row += 1
    
    if row > 1:
        table_column_headers = [ { 'header': name } for name in headings ]
        sheet.add_table(0, 0, row - 1, len(table_column_headers) - 1, { 
            'name': skey,
            'banded_rows': True,
            'columns': table_column_headers,
        })
    
    return row

In [ ]:
worksheets = dict()

for key, system in systems.items():
    title = re.sub(r'[\:\[\]*?/\\]', '_', system['name'])
    length = min(25, len(title))
    t = key.replace('INTF','') + ' ' + title[:length]
    if t.lower() in worksheets.keys():
        t = key.replace('INTF','') + ' ' + title[max(0, len(title) - 25):]
    worksheet = workbook.add_worksheet(t)
    worksheets[t.lower()] = worksheet
    rows = fill_worksheet(spod, key, system, worksheet)
    print(f"{key}: {system['name']} -> {t} with {rows} rows")

## Summary sheet

In [ ]:
summary.write(0, 0, "Summary", title_format)

row = 2
summary.write(row, 0, 'Key', column_head_format)
summary.write(row, 1, 'Name', column_head_format)
summary.write(row, 2, 'Mapped', column_head_format)
summary.write(row, 3, 'Total', column_head_format)
summary.write(row, 4, 'Tables', column_head_format)
summary.write(row, 5, 'Sheet Link', column_head_format)

row = 3
for skey, system in systems.items():
    summary.write(row, 0, skey)
    summary.write(row, 1, system['name'])
    
    columns = list(filter(lambda c: c['database-id+'] == skey, spod['columns'].values()))
    mapped = list(filter(lambda c: len(c['attributesmapped']) > 0, columns))
    summary.write(row, 2, len(mapped))
    summary.write(row, 3, len(columns))
    
    summary.write(row, 4, len(system['tables+']))
    
    _, ws = next(iter(filter(lambda t: skey[4:] in t[0], worksheets.items())))
    summary.write_url(row, 5, f"internal:'{ws.get_name()}'!A1", string=f'Sheet {skey[4:]}')
    
    row += 1

## Styling

In [ ]:
summary.set_column(0, 0, 20)
summary.set_column(1, 1, 60)
summary.set_column(2, 5, 15)

## Write Excel file

In [ ]:
version_file = Path(LIBRARY, 'versons.json')
if version_file.is_file():
    with open(version_file, 'r') as src:
        version = json.load(src)
else:
    version = { 'TOOLVERSION': '?.?' }

In [ ]:
workbook.set_properties({
    'title':    f"{spod['model']['name']}",
    'subject':  'mapping',
    'author':   f"Excel Mapping Publisher {version['TOOLVERSION']}",
#    'manager':  'D',
#    'company':  'of Wolves',
    'category': 'export',
    'keywords': 'Information Model, Data Models',
    'comments': f"generated with {version['TOOLVERSION']} from model {spod['_imprint_']['Modelversion']}",
    'status':   'Draft',
    'revision': spod['_imprint_'].get('git')
})

In [ ]:
workbook.close()
print(f"Wrote {xlsx_destination}")

# Visually verify

In [ ]:
import subprocess

In [ ]:
# Open Excel on MacOS
r = subprocess.run(['open', '-a', 'Microsoft Excel', xlsx_destination], shell=False) # capture_output=False, stderr=subprocess.DEVNULL)

In [ ]:
# MacOS Preview
#r = subprocess.run(['qlmanage', '-x', '-p', xlsx_destination], shell=False) # capture_output=False, stderr=subprocess.DEVNULL)

In [ ]:
import pandas
excel_data_df = pandas.read_excel(DESTINATION, sheet_name='Mapping')

In [ ]:
from IPython.display import display, HTML
display(excel_data_df)